In [1]:
import torch
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime


# Load my functions
from utils import minmax_normalise_tensor, midpoint_to_box, upscale_tensor, box_tensor_2_mid_points
from metrics import rmse
from covariance import covariance_function, spatial_covariance_function, pixel_intensity_covariance_function, aggregate_columns_rows_subsetcase
from predictive import predictive_distribution_cholesky_correction

# Avoid scientific numeric notation
torch.set_printoptions(sci_mode = False)

/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIxEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  Referenced from: <67CD63CE-57E0-341F-B3B8-78729B03D2B3> /Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/image.so
  Expected in:     <18497461-1393-3DF8-BED0-DC986FDB1051> /Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warn(f"Failed to load image Python extension: {e}")


# Load scene

Scene target tensor: [N, C, H, W]  

with channels: 
- `[:, 0, :, :]` bed
- `[:, 1, :, :]` surface
- `[:, 2, :, :]` thickness
- `[:, 3, :, :]` mask
- `[:, 4, :, :]` firn
- `[:, 5, :, :]` errorbed

and polar stereographic coordinates
- `[:, 6, :, :]` y
- `[:, 7, :, :]` x

# Language/terminology
- low resolution input tensor
- target tensor
  - inferred
  - ground truth
- auxiliary tensor

# Easy case: 60 pixel HW images
- aligning grids: Aux grid is the same 
- perfect upscale: 1, 2, 3, 4, 5, 6 times upscale fix perfectly

In [122]:
# scene_bed_tensor = torch.load('./torch_data/DOMEC_bed_scenes_60pixel.pt')
# scene_bed_tensor = torch.load('./torch_data/TRANSANT_bed_scenes_60pixel.pt')

training_tensor = torch.load('./torch_data/TRANSANT_experiments/TRANSANT_training_tensor.pt')

print(training_tensor.shape)

torch.Size([300, 8, 60, 60])


In [121]:
dataloader = torch.utils.data.DataLoader(scene_bed_tensor, batch_size = 8, shuffle = False)

# Will for through all data - num depends on the batch_size
for i in dataloader:
    print(i.shape)

torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
torch.Size([8, 8, 60, 60])
t

In [222]:
class SuperresolutionGP:
    def __init__(self, train_tensor):
        self.n_train = train_tensor.shape[0]
        self.hr_hw = train_tensor.shape[-1] # height width
        self.up_factors = np.array([2, 3, 4, 5, 6]) # hardcoded
        self.n_up_factors = len(self.up_factors)
        self.up_lr_hw = (np.repeat(self.hr_hw, self.n_up_factors) / self.up_factors).astype(int)

        # Tensors
        self.train_bed_ground_truth = train_tensor[:, 0, :, :].unsqueeze(1) # channel 0 contains bed, retain channel dim
        self.train_sur_hr_aux = train_tensor[:, 1, :, :].unsqueeze(1) # channel 1 contains surface, retain channel dim
        # Upscale and retain list of tensors (varying dimensionality)
        self.train_bed_lr = self.upscale(self.train_bed_ground_truth)

        # Baseline
        self.train_baseline_predictions = self.bilinear_interpolation_baseline(self.train_bed_lr) # torch.Size([5, 300, 1, 60, 60])

        # normalise

    def upscale(self, bed_ground_truth):
        # Upscale (Increase scale of each pixel, reduce resolution) to articially generate low-res. input. controlled experiment.
        # Empty list of tensors (varying dims - can't stack tensors)
        train_lr = []
        for u in self.up_factors:
            # define upscaling function with torch https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html, default settings
            upscaling_function = torch.nn.AvgPool2d(kernel_size = u)
            # appending is in-place: no reassignment needed
            train_lr.append(upscaling_function(bed_ground_truth))
        return train_lr
    
    def bilinear_interpolation_baseline(self, bed_lr):
        # bed_lr is a list of tensors

        # Create one target grid: outer boundries of each scene are the same for both hr and lr: [-1, 1]
        d = torch.tensor(np.linspace(start = (-1.0 + (2/self.hr_hw)/2) , stop = (1.0 - (2/self.hr_hw)/2), num = self.hr_hw))
        meshx, meshy = torch.meshgrid((d, d), indexing = "xy") # create mesh
        target_grid = torch.stack((meshx, meshy), 2) # x,y order
        target_grid = target_grid.unsqueeze(0) # add batch dim: torch.Size([1, self.hr_hw, self.hr_hw, 2])

        # Create empty placeholder tensor of shape [Up, N, C = 1, H, W]
        bed_hr = torch.empty(size = (0, bed_lr[0].shape[0], 1, self.hr_hw, self.hr_hw))

        # Simple baseline in bilinear interpolation using torch https://pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html 
        for u_index, u in enumerate(self.up_factors):
            # copy target grid n times
            n = bed_lr[u_index].shape[0]
            n_target_grids = torch.tile(target_grid, dims = (n, 1, 1, 1))

            hr_bilinear = torch.nn.functional.grid_sample(bed_lr[u_index].float(), n_target_grids.float(), mode = 'bilinear', padding_mode = 'border', align_corners = False)
            bed_hr = torch.cat((bed_hr, hr_bilinear.unsqueeze(0)), dim = 0) # generate explicit first dim for all up_factors

        return(bed_hr)
    
    def rmse(self, ground_truth, predictions):
        n = predictions.shape[0]
        n_ground_truth = torch.tile(ground_truth.unsqueeze(0), dims = (n, 1, 1, 1, 1))
        diff = torch.sub(n_ground_truth, predictions) # subtract elementwise
        print(diff.shape)

        

In [223]:
srgp = SuperresolutionGP(train_tensor = training_tensor)
srgp.rmse(srgp.train_bed_ground_truth, srgp.train_baseline_predictions)

torch.Size([5, 300, 1, 60, 60])


In [39]:
def run_experiment(list_of_covariance_functions, scene_bed_tensor, n_scenes, lambda_p_value = 0.4, lambda_s_value = 0.3):

    # n_scenes (int): number of scenes in dataset to iterate over. 400 is the maximum here
     
    n_covar_functions = len(list_of_covariance_functions)
    # Define upscaling factor. We use 60 pixel images so that upscaling is not compromised for 2 to 6.
    up_factor_list = [2, 3, 4, 5, 6]
    # Generate list of column names
    column_names = [("up_") + str(x) for x in up_factor_list] * n_covar_functions

    # Empty dataframes for losses: rows represent scenes (to guide further investigation) and columns represent upscaling factors
    proposed_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)
    baseline_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)
    # NLL only makes sense for probabilistic methods
    proposed_nll_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)

    # Log marginal likelihood placeholder
    proposed_lml_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)


    # for each upscaling_factor
    for u_index, u in enumerate(up_factor_list):
        # for each scene (convert to batches later)
        for i in range(0, n_scenes):
            # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 60, 60])
            target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)
            # Extract target height-width: last dimension (since H == W)
            target_hw = target_ground_truth.shape[-1]
            lr_hw = int(target_hw / u)
                
            # Upscale (Increase scale of each pixel, reduce resolution) to generate low-res. input
            lr_bed = upscale_tensor(target_ground_truth, upscaling_factor = u)

            ### BASELINE ###
            # For torch grid resample function: Normalised grid as image input [N, C, H, W] where N = 1 and C = 1. 
            # H_in and W_in are implicit: corners of midpoints are assumed to me -1, -1 (top left) and 1, 1 (bottom right).
            # Always first dim
            lr_input_grid = lr_bed.unsqueeze(0).unsqueeze(0)

            # Assuming boundries are the same for both: outer boundries are [-1, 1] for both hr and lr
            d = torch.tensor(np.linspace(start = (-1.0 + (2/target_hw)/2) , stop = (1.0 - (2/target_hw)/2), num = target_hw))
            meshx, meshy = torch.meshgrid((d, d), indexing = "xy")
            # x,y order
            target_grid = torch.stack((meshx, meshy), 2)
            target_grid = target_grid.unsqueeze(0) # add batch dim
                
            # Border works much better than zero: Since we sample at a higher resolution than the input we sample e.g. left of the leftmost HR location
            hr_bilinear = torch.nn.functional.grid_sample(lr_input_grid.float(), target_grid.float(), mode = 'bilinear', padding_mode = 'border', align_corners = False)
            hr_bilinear = hr_bilinear.squeeze()

            ### BASELINE LOSS ###
            baseline_rmse_df.iloc[i, u_index] = rmse(hr_bilinear, target_ground_truth.squeeze()).numpy().item()

            ### PROPOSED ###
            # Normalisation of high-resolution auxiliary channel to compute the base_covariance
            hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

            # Iterate through covar functions
            for c_index, c in enumerate(list_of_covariance_functions):

                ### Base covariance ###
                base_covariance = c(hr_aux, lambda_p = lambda_p_value, lambda_s = lambda_s_value)
                k_ah_al_tensor, k_al_al_tensor = aggregate_columns_rows_subsetcase(base_covariance = base_covariance, u = u)

                ### MEAN RECONSTRUCTION ###
                hr_mean_inferred, hr_covariance_inferred, lml = predictive_distribution_cholesky_correction(lr_bed.unsqueeze(0), base_covariance, k_ah_al_tensor, k_al_al_tensor, 
                                                                                noise = torch.tensor(0.05), mu = torch.tensor(0.5))

                # Save log marginal likelihood
                proposed_lml_df.iloc[i, (u_index + (c_index * 5))] = lml

                ### LOSS ###
                # inplace mutation of row i and column u (upscale_factor) of df
                proposed_rmse_df.iloc[i, (u_index + (c_index * 5))] = rmse(hr_mean_inferred, target_ground_truth).numpy().item()

                # NLL; store items not tensors in df
                # extract variances (on diagonal) from covariance matrix and reshape to square
                hr_variance_inferred = torch.diagonal(hr_covariance_inferred).reshape(1, target_hw, -1)
                proposed_nll_df.iloc[i, (u_index + (c_index * 5))] = torch.nn.functional.gaussian_nll_loss(hr_mean_inferred, target_ground_truth, hr_variance_inferred, full = False, eps = 1e-06, reduction = 'mean').numpy().item()
    
    return(proposed_rmse_df, baseline_rmse_df, proposed_nll_df, proposed_lml_df)

In [66]:
list_of_covar_functions = [covariance_function, spatial_covariance_function]
names_list_of_covar_functions = ["Composite kernel", "Spatial kernel"]
num_scenes = 20 # 400 is max

proposed_rmse_df, baseline_rmse_df, proposed_nll_df, proposed_lml_df = run_experiment(list_of_covariance_functions = list_of_covar_functions, 
                                                                                      scene_bed_tensor = scene_bed_tensor, 
                                                                                      n_scenes = num_scenes,
                                                                                      lambda_p_value = 2.0)

In [67]:
def visualise_results(proposed_rmse_df, baseline_rmse_df, proposed_nll_df, proposed_lml_df, n_scenes, domain_name, names_list_of_covariance_functions):

    # RMSE
    fig = go.Figure()
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y =  baseline_rmse_df.mean(), mode = 'lines+markers', name = "Bilinear baseline"))
    for c_index, c in enumerate(names_list_of_covar_functions):
        fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_rmse_df.iloc[:, np.asarray(range(0, 5)) + (c_index * 5)].mean(), 
                                 mode = 'lines+markers', name = "Proposed algorithm: {}".format(c)))
    fig.update_layout(title = 'Reconstruction loss [RMSE] of proposed vs. baseline - {} scenes near domain {}'.format(n_scenes, domain_name))
    fig.update_xaxes(title_text = 'Upscaling factor')
    fig.update_yaxes(title_text = 'RMSE')
    fig.show()

    # NLL
    fig = go.Figure()
    for c_index, c in enumerate(names_list_of_covariance_functions):
        fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_nll_df.iloc[:, np.asarray(range(0, 5)) + (c_index * 5) ].mean(), 
                                 mode = 'lines+markers', name = "Proposed algorithm: {}".format(c)))
    fig.update_layout(title = "Reconstruction loss [NLL]")
    fig.update_xaxes(title_text = 'Upscaling factor')
    fig.update_yaxes(title_text = 'NLL')
    fig.show()

    # LML
    fig = go.Figure()
    for c_index, c in enumerate(names_list_of_covariance_functions):
        fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_lml_df.iloc[:, np.asarray(range(0, 5)) + (c_index * 5) ].mean(), 
                                 mode = 'lines+markers', name = "Proposed algorithm: {}".format(c)))
    fig.update_layout(title = "Reconstruction Log marginal likelihood [LML] (larger is better)")
    fig.update_xaxes(title_text = 'Upscaling factor')
    fig.update_yaxes(title_text = 'LML')
    fig.show()

In [68]:
domain_name = "Dome C"
names_list_of_covar_functions = ["Composite kernel", "Spatial kernel"]

visualise_results(proposed_rmse_df, baseline_rmse_df, proposed_nll_df, proposed_lml_df, num_scenes, domain_name, names_list_of_covar_functions)

## Log Marginal likelihood  

Hyperparatemers:
lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0
- lambda_s controls the distance cut-off (Euclidean) where values may still influence the prediction.

Optimise:
- For a random batch/all
- 

In [51]:
lambda_p_values = np.linspace(0.1, 5, 10)
lambda_s_values = np.linspace(1, 1, 10)

list_of_covar_functions = [covariance_function]
num_scenes = 10 # 400 is max

_, _, _, proposed_lml_df = run_experiment(list_of_covariance_functions = list_of_covar_functions,
                                          scene_bed_tensor = scene_bed_tensor, n_scenes = num_scenes)
mean_lml = []
for p in lambda_p_values:
    for s in lambda_s_values:
        _, _, _, proposed_lml_df = run_experiment(list_of_covariance_functions = list_of_covar_functions, 
                                              scene_bed_tensor = scene_bed_tensor, 
                                              n_scenes = num_scenes,
                                              lambda_p_value = p,
                                              lambda_s_value = s)
        mean_lml.append(proposed_lml_df.mean().mean())
    

In [58]:
np.array(mean_lml).reshape(10, 10)

array([[117.42055893, 117.42055893, 117.42055893, 117.42055893,
        117.42055893, 117.42055893, 117.42055893, 117.42055893,
        117.42055893, 117.42055893],
       [145.26395813, 145.26395813, 145.26395813, 145.26395813,
        145.26395813, 145.26395813, 145.26395813, 145.26395813,
        145.26395813, 145.26395813],
       [146.60996399, 146.60996399, 146.60996399, 146.60996399,
        146.60996399, 146.60996399, 146.60996399, 146.60996399,
        146.60996399, 146.60996399],
       [146.96588196, 146.96588196, 146.96588196, 146.96588196,
        146.96588196, 146.96588196, 146.96588196, 146.96588196,
        146.96588196, 146.96588196],
       [147.10925842, 147.10925842, 147.10925842, 147.10925842,
        147.10925842, 147.10925842, 147.10925842, 147.10925842,
        147.10925842, 147.10925842],
       [147.1806427 , 147.1806427 , 147.1806427 , 147.1806427 ,
        147.1806427 , 147.1806427 , 147.1806427 , 147.1806427 ,
        147.1806427 , 147.1806427 ],
       [14

In [59]:
fig = go.Figure(data = go.Contour(z = np.array(mean_lml).reshape(10, 10), 
                                  ))
fig.show()

In [60]:
# class GP_superresolution:
fig = go.Figure(data = go.Contour(z = np.array(mean_lml).reshape(10, 10), 
                                  x = lambda_s_values, # horizontal, inner loop
                                  y = lambda_p_values # vertical,  outer loop 
                                  ))

fig.update_layout(title = "LML contours as a function of hyperparameters",
    xaxis_title = "lambda_s",
    yaxis_title = "lambda_p",
    height = 600)

fig.show()

In [6]:
"""
### LINE_BY_LINE VERSION ###

# Define upscaling factor
# 60 pixel images so that upscaling is easy for 0 to 6
up_factor_list = [2, 3, 4, 5, 6]
# up_factor_list = [5]
n_scenes = 400 # max 400, 200

# For loop first and wrap up into a function once it works
proposed_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])
baseline_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])
proposed_nll_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])

proposed_lml_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])

# for each scene (or fist n_scenes)
for u_index, u in enumerate(up_factor_list):
    for i in range(0, n_scenes):
        # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 60, 60])
        target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)
        # Extract target height-width: last dimension (since H == W)
        target_hw = target_ground_truth.shape[-1]
        lr_hw = int(target_hw / u)
        
        # Upscale (Increase scale of each pixel, reduce resolution) to generate low-res. input
        lr_bed = upscale_tensor(target_ground_truth, upscaling_factor = u)

        # Normalisation of high-resolution auxiliary channel to compute the base_covariance
        hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

        ### Base covariance ###
        base_covariance = covariance_function(hr_aux)
        k_ah_al_tensor, k_al_al_tensor = aggregate_columns_rows_subsetcase(base_covariance = base_covariance, u = u)

        ### MEAN RECONSTRUCTION ###
        # hr_inferred = predictive_mean(lr_bed.unsqueeze(0), k_ah_al_tensor, k_al_al_tensor, noise = 0.05, mu = 0.5)
        # hr_covariance_inferred = predictive_variance(base_covariance, k_ah_al_tensor, k_al_al_tensor, noise = 0.05)

        hr_mean_inferred, hr_variance_inferred, lml = predictive_distribution_cholesky_correction(lr_bed.unsqueeze(0), base_covariance, k_ah_al_tensor, k_al_al_tensor, 
                                                                         noise = torch.tensor(0.05), mu = torch.tensor(0.5))

        # Save log marginal likelihood
        proposed_lml_df.iloc[i, u_index] = lml

        ### LOSS ###
        # inplace mutation of row i and column u (upscale_factor) of df
        proposed_rmse_df.iloc[i, u_index] = rmse(hr_mean_inferred, target_ground_truth).numpy().item()

        # NLL
        # torch.nn.functional.gaussian_nll_loss(input, target, var, full=False, eps=1e-06, reduction='mean')

        ### BASELINE ###
        # For torch grid resample function: Normalised grid as image input [N, C, H, W] where N = 1 and C = 1. 
        # H_in and W_in are implicit: corners of midpoints are assumed to me -1, -1 (top left) and 1, 1 (bottom right).
        # Always first dim
        lr_input_grid = lr_bed.unsqueeze(0).unsqueeze(0)

        # Assuming boundries are the same for both: outer boundries are [-1, 1] for both hr and lr
        d = torch.tensor(np.linspace(start = (-1.0 + (2/target_hw)/2) , stop = (1.0 - (2/target_hw)/2), num = target_hw))
        meshx, meshy = torch.meshgrid((d, d), indexing = "xy")
        # x,y order
        target_grid = torch.stack((meshx, meshy), 2)
        target_grid = target_grid.unsqueeze(0) # add batch dim
        
        # Border works much better than zero: Since we sample at a higher resolution than the input we sample e.g. left of the leftmost HR location
        hr_bilinear = torch.nn.functional.grid_sample(lr_input_grid.float(), target_grid.float(), mode = 'bilinear', padding_mode = 'border', align_corners = False)
        hr_bilinear = hr_bilinear.squeeze()

        ### BASELINE LOSS ###
        baseline_rmse_df.iloc[i, u_index] = rmse(hr_bilinear, target_ground_truth.squeeze()).numpy().item()
"""

'\n### LINE_BY_LINE VERSION ###\n\n# Define upscaling factor\n# 60 pixel images so that upscaling is easy for 0 to 6\nup_factor_list = [2, 3, 4, 5, 6]\n# up_factor_list = [5]\nn_scenes = 400 # max 400, 200\n\n# For loop first and wrap up into a function once it works\nproposed_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\nbaseline_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\nproposed_nll_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\n\nproposed_lml_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\n\n# for each scene (or fist n_scenes)\nfor u_index, u in enumerate(up_factor_list):\n    for i in range(0, n_scenes):\n        # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 60, 60])\n        tar

- Rewrite for batches
- Cholesky implementation for correction
- Vary the covariance function

In [7]:
# Check covariance properies
# Hermitian: symmetry check
# print(k_al_al_tensor == k_al_al_tensor.mT)
# non-negative eigenvalue check
# torch.linalg.eigvalsh(k_al_al_tensor)

In [8]:
# LML mainly for HP opt.
fig = go.Figure()
fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y =  baseline_rmse_df.mean(), mode = 'lines+markers', name = "Bilinear baseline"))
fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_rmse_df.mean(), mode = 'lines+markers', name = "Bayesian Fusion using auxiliary surface data"))
fig.update_layout(title = "Reconstruction loss [RMSE] compared to baseline - 400 scenes near Dome C")
fig.update_xaxes(title_text = 'Upscaling factor')
fig.update_yaxes(title_text = 'RMSE')
fig.show()

In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_lml_df.mean(), mode = 'lines+markers', name = "Bayesian Fusion using auxiliary surface data"))
fig.update_layout(title = "Reconstruction Log marginal likelihood (larger is better)")
fig.update_xaxes(title_text = 'Upscaling factor')
fig.update_yaxes(title_text = 'LML')
fig.show()

- Determine size of scense based on HPs 

## Cholesky

In [10]:
# Cholesky
a = torch.randn(3, 3)
# Returns a view of this tensor with the last two dimensions transposed. Thus this is suitable when first dims are batches etc.
a = a @ a.mT + 1e-3 # make symmetric positive-definite
print(a)
l = torch.linalg.cholesky(a)
print(l @ l.mT)

reg_inv = torch.linalg.inv(a)
print(reg_inv)

chol_inv = torch.cholesky_inverse(torch.linalg.cholesky(a))
print(chol_inv)

tensor([[ 2.2670,  0.2140,  2.1087],
        [ 0.2140,  1.8259, -0.8811],
        [ 2.1087, -0.8811,  3.4343]])
tensor([[ 2.2670,  0.2140,  2.1087],
        [ 0.2140,  1.8259, -0.8811],
        [ 2.1087, -0.8811,  3.4343]])
tensor([[ 1.6235, -0.7662, -1.1935],
        [-0.7662,  0.9867,  0.7236],
        [-1.1935,  0.7236,  1.2096]])
tensor([[ 1.6235, -0.7662, -1.1935],
        [-0.7662,  0.9867,  0.7236],
        [-1.1935,  0.7236,  1.2096]])


In [11]:
a.shape

torch.Size([3, 3])

# Understanding torch.grid_sample() and torch.meshgrid()

- [Blog post about i,j (Matrix) and x,y (Cartesian) indexing](https://sparrow.dev/numpy-meshgrid/)
- [ptrblck pytorch discussion on grid_sample()](https://discuss.pytorch.org/t/solved-torch-grid-sample/51662)

Derive same grid from (a) ij Matrix indexing

In [12]:
i = torch.tensor([1, 2, 3]) # called x but should be y
j = torch.tensor([4, 5, 6])

# if i == j order does not matter in input
ii, jj = torch.meshgrid(i, j, indexing = 'ij')

### i == y ###
# ii are the grid of all y-values (rows)
print(ii)

# The first row of y-values are on the same row thus have contants value
print(ii[0, :])

### j == x ###
print(jj)

# Change to x, y order
xy_grid = torch.stack((jj, ii), 2)

print(xy_grid)

tensor([[1, 1, 1],
        [2, 2, 2],
        [3, 3, 3]])
tensor([1, 1, 1])
tensor([[4, 5, 6],
        [4, 5, 6],
        [4, 5, 6]])
tensor([[[4, 1],
         [5, 1],
         [6, 1]],

        [[4, 2],
         [5, 2],
         [6, 2]],

        [[4, 3],
         [5, 3],
         [6, 3]]])


Derive same grid from (b) xy Cartesian indexing. xy will be the default in futture pytorch versions.

In [13]:
# Other way to derive at the same result
y = torch.tensor([1, 2, 3]) 
x = torch.tensor([4, 5, 6])

xx, yy = torch.meshgrid(x, y, indexing = 'xy')

xy_grid = torch.stack((xx, yy), 2)

print(xy_grid)

tensor([[[4, 1],
         [5, 1],
         [6, 1]],

        [[4, 2],
         [5, 2],
         [6, 2]],

        [[4, 3],
         [5, 3],
         [6, 3]]])


### Example from Patrick

In [14]:
input = torch.arange(4*4).view(1, 1, 4, 4).float()
print(input)

# Create grid to upsample input
d = torch.linspace(-1, 1, 8)

### Using Cartesian xy indexing ###
# both grid axis are the same so order does not matter in this case.
xx, yy = torch.meshgrid((d, d), indexing = "xy")
# x,y order of last two dimensions of grid
grid = torch.stack((xx, yy), 2)
grid = grid.unsqueeze(0) # add batch dim

# 09/2023 default is align_corners = False
output = torch.nn.functional.grid_sample(input, grid, align_corners = True)
print(output)

### Using Matrix ij indexing ###
# This is the default and was used in the referance example I am trying to reporduce
# ii corresponds to rows in matrix indexing, so y-axis in Cartesian indexing
ii, jj = torch.meshgrid((d, d), indexing = "ij")
# Same: ii, jj = torch.meshgrid((d, d))
# Reverse order to match x, y Cartesian ordering: jj == xx, ii == yy in previous example
grid = torch.stack((jj, ii), 2)
grid = grid.unsqueeze(0) # add batch dim

# 09/2022 default is align_corners = False
output = torch.nn.functional.grid_sample(input, grid, align_corners = True)
print(output)

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])
tensor([[[[ 0.0000,  0.4286,  0.8571,  1.2857,  1.7143,  2.1429,  2.5714,
            3.0000],
          [ 1.7143,  2.1429,  2.5714,  3.0000,  3.4286,  3.8571,  4.2857,
            4.7143],
          [ 3.4286,  3.8571,  4.2857,  4.7143,  5.1429,  5.5714,  6.0000,
            6.4286],
          [ 5.1429,  5.5714,  6.0000,  6.4286,  6.8571,  7.2857,  7.7143,
            8.1429],
          [ 6.8571,  7.2857,  7.7143,  8.1429,  8.5714,  9.0000,  9.4286,
            9.8571],
          [ 8.5714,  9.0000,  9.4286,  9.8571, 10.2857, 10.7143, 11.1429,
           11.5714],
          [10.2857, 10.7143, 11.1429, 11.5714, 12.0000, 12.4286, 12.8571,
           13.2857],
          [12.0000, 12.4286, 12.8571, 13.2857, 13.7143, 14.1429, 14.5714,
           15.0000]]]])
tensor([[[[ 0.0000,  0.4286,  0.8571,  1.2857,  1.7143,  2.1429,  2.5714,
            3.0000],
          [

### Reproduce any grid with grid_sample()

From documentation:
"For example, values x = -1, y = -1 is the left-top pixel of input, and values x = 1, y = 1 is the right-bottom pixel of input." However these are the boarders of the pixels, not the midpoints. 

In [15]:
dims = 8
random_values = torch.rand(size = (dims, dims)) * 100
print(random_values)

tensor([[84.3973, 10.6304,  5.3981,  8.6185,  2.1829, 66.3769,  5.9031, 15.6914],
        [55.8399, 33.9762, 49.1713, 76.7333, 29.9037, 37.6345, 89.8326, 73.2113],
        [95.2115, 84.9728, 31.1442, 48.1289, 96.1435, 30.1152, 48.1031, 35.8865],
        [16.5478, 54.7446, 36.5655,  9.5899,  6.2165, 44.4285,  5.7303, 45.1680],
        [44.1684,  0.9103, 19.6046, 21.7739, 83.9929, 45.8482, 89.3310, 30.6984],
        [69.8361, 24.1347,  5.3156, 30.0250, 86.2330, 43.2686, 98.0689, 80.8819],
        [47.5328, 76.4295,  4.1695, 53.2799, 35.8222, 21.5390, 17.7300, 71.5801],
        [52.3480, 11.5171, 28.3725, 50.6455, 39.0819, 31.5751, 96.9057, 65.0751]])


In [16]:
# 2.0 is the range, devide by 2 because we have centroids (which are at the midpoint of each cell)
d = torch.tensor(np.linspace(start = (-1.0 + (2.0 / dims) / 2) , stop = (1.0 - (2.0 / dims) / 2), num = dims))
meshx, meshy = torch.meshgrid((d, d), indexing = "xy")
grid = torch.stack((meshx, meshy), 2)
grid = grid.unsqueeze(0) # add batch dim

torch.nn.functional.grid_sample(random_values.unsqueeze(0).unsqueeze(0).float(), grid.float())
# same as torch.nn.functional.grid_sample(random_values.unsqueeze(0).unsqueeze(0).float(), grid.float(), align_corners = False, mode = 'bilinear', padding_mode = 'border')
# Padding mode does not matter though


/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torch/nn/functional.py:4227: UserWarning:

Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.



tensor([[[[84.3973, 10.6304,  5.3981,  8.6185,  2.1829, 66.3769,  5.9031,
           15.6914],
          [55.8399, 33.9762, 49.1713, 76.7333, 29.9037, 37.6345, 89.8326,
           73.2113],
          [95.2115, 84.9728, 31.1442, 48.1289, 96.1435, 30.1152, 48.1031,
           35.8865],
          [16.5478, 54.7446, 36.5655,  9.5899,  6.2165, 44.4285,  5.7303,
           45.1680],
          [44.1684,  0.9103, 19.6046, 21.7739, 83.9929, 45.8482, 89.3310,
           30.6984],
          [69.8361, 24.1347,  5.3156, 30.0250, 86.2330, 43.2686, 98.0689,
           80.8819],
          [47.5328, 76.4295,  4.1695, 53.2799, 35.8222, 21.5390, 17.7300,
           71.5801],
          [52.3480, 11.5171, 28.3725, 50.6455, 39.0819, 31.5751, 96.9057,
           65.0751]]]])